[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Host_Prog/Intro_OS/Intro_OS.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to Operating Systems

Every benchmark you've run in this curriculum — the NumPy timings in [Intro to Python](../../Intro_Programming/Intro_Python/Intro_Python.ipynb), the GPU transfers in [Intro to GPU Systems](../../Intro_GPU/Intro_GPU.ipynb) — happened *on top of* an operating system quietly scheduling, paging, and buffering underneath you. This workshop makes that layer visible: processes, memory, concurrency, and the shell.

## 0. Introduction

An OS does three jobs:

1. **Abstraction** — files instead of disk blocks, processes instead of CPU time slices.
2. **Multiplexing** — hundreds of programs sharing a few cores and one memory.
3. **Protection** — your buggy pointer arithmetic ([Intro to C](../../Intro_Programming/Intro_C.ipynb) §5) crashes *your* process, not the machine.

We poke at all three with live code.

## 1. Pre-requisites

- [Intro to Python](../../Intro_Programming/Intro_Python/Intro_Python.ipynb); [Intro to C](../../Intro_Programming/Intro_C.ipynb) for the memory sections.
- **A Unix-like environment**: Linux, macOS, WSL on Windows, or Google Colab (which is Linux underneath — every cell here runs there). Windows-native Python will fail on the `/proc` cells.

---
### 🕐 Session 1 of 4 — *Processes & the Kernel* (~35 min)
**Goal:** see what a process is, watch syscalls happen, and understand user vs kernel mode.
**Feeds into:** Session 2 (memory & scheduling).

---

<details><summary>🎓 <b>Teacher notes — Session 1: Processes & the Kernel</b></summary>

**Anchor the whole session on one sentence:** a process is a program "caught in the act" — code plus its memory, open files, and a kernel-assigned identity (the PID). Everything else in Session 1 is elaboration on that.

**The process-tree cell is the best demo in the session — don't rush past it.** Students see their *own* kernel process's ancestry back to PID 1/systemd, walking `/proc/{pid}/stat` field 4 (parent PID) by hand. It's concrete proof that "every process has a parent, back to one root" isn't an abstraction — it's a chain they can print. If running this outside the sandboxed environment shown here, the chain will differ (no `claude`/`timeout` wrapper), which is itself worth pointing out: the *shape* (some shell, eventually PID 1) is universal; the specific names are not.

**`strace` is the syscall doorbell made audible.** The point isn't the specific counts — it's that a do-nothing `print('hi')` program makes ~300 syscalls just to start up (dynamic linking, memory mapping, terminal ioctls). Connect this explicitly to the "minimize syscalls" advice students will see again in real-time DSP contexts: every syscall is a trip through kernel mode with unpredictable latency, which is why buffered I/O and batched reads matter for anything with a real-time deadline.

**User vs. kernel mode:** frame it as a *privilege* boundary, not a physical one — same CPU, different mode bit. Your process can compute freely but must ring the syscall doorbell for anything touching hardware (files, memory beyond what you already have, network). This is also *why* a crash in your process can't (normally) crash the machine — protection is enforced at exactly this boundary, which sets up the "protection" bullet from the intro section.

**If short on time:** the process-tree cell and the strace cell together carry ~80% of the session's value; the PID/getuid/getcwd cell is a warm-up, not essential to dwell on.
</details>

## 2. Processes

💡 **Intuition.** A *program* is a file; a *process* is that program **caught in the act**: code plus its memory, open files, and a kernel-side identity (the PID). The kernel is the only code with full hardware access; your process must *ask* for anything beyond arithmetic — every file read, print, and allocation is a **syscall**, a controlled doorbell into the kernel.

In [1]:
import os, sys

print("my PID:", os.getpid())
print("my parent's PID:", os.getppid(), "(the kernel/jupyter that launched me)")
print("running as user:", os.getuid())
print("current working dir:", os.getcwd())

my PID: 1893734
my parent's PID: 1893724 (the kernel/jupyter that launched me)
running as user: 1000
current working dir: /tmp/claude-1000/-home-jibby2k1-Projects-SPS-Curriculum/fa83eb7b-163e-43b8-89a1-d2f6ef7e8f87/scratchpad


### 2.1. The Process Tree

Every process is spawned by another — back to PID 1. Your notebook kernel is a child of the Jupyter server, which is a child of your shell…

In [2]:
import subprocess
# walk our own ancestry via /proc
pid = os.getpid()
chain = []
while pid > 1:
    with open(f"/proc/{pid}/comm") as f:
        chain.append((pid, f.read().strip()))
    with open(f"/proc/{pid}/stat") as f:
        pid = int(f.read().split()[3])          # field 4 = parent PID
chain.append((1, "init/systemd"))
for depth, (p, name) in enumerate(reversed(chain)):
    print("  " * depth + f"└─ {p}: {name}")

└─ 1: init/systemd
  └─ 4747: systemd
    └─ 72324: gnome-terminal-
      └─ 1710564: bash
        └─ 1712198: claude
          └─ 1893719: bash
            └─ 1893721: timeout
              └─ 1893724: python
                └─ 1893734: python


### 2.2. Watching Syscalls

`strace` prints every doorbell a process rings. Even a do-nothing program makes dozens of syscalls just to start up.

In [3]:
r = subprocess.run(["strace", "-c", "python3", "-c", "print('hi')"],
                   capture_output=True, text=True)
if r.returncode == 0:
    print(r.stderr[-1400:])          # strace summarizes to stderr
else:
    print("strace not installed — on your own machine: sudo apt install strace")

     14         3 lseek
  1.61    0.000002           1         2           munmap
  1.61    0.000002           2         1           getcwd
  1.61    0.000002           1         2         1 readlink
  1.61    0.000002           2         1           gettid
  0.81    0.000001           0        66           rt_sigaction
  0.81    0.000001           0        11        11 ioctl
  0.00    0.000000           0         6           mprotect
  0.00    0.000000           0         2           pread64
  0.00    0.000000           0         1         1 access
  0.00    0.000000           0         1           execve
  0.00    0.000000           0         4           fcntl
  0.00    0.000000           0         1           getuid
  0.00    0.000000           0         1           getgid
  0.00    0.000000           0         1           geteuid
  0.00    0.000000           0         1           getegid
  0.00    0.000000           0         1           arch_prctl
  0.00    0.000000           0   

Read the table: `read`, `write`, `mmap`, `openat` — *file access and memory mapping dominate even a hello-world*. This is why 'minimize syscalls' is a real optimization strategy in real-time signal processing.

---
### 🕐 Session 2 of 4 — *Memory & Scheduling* (~35 min)
**Goal:** understand virtual memory and the scheduler — and why your benchmark numbers wobble.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (concurrency).

---

<details><summary>🎓 <b>Teacher notes — Session 2: Memory & Scheduling</b></summary>

**The "lazy allocation" demo is the single most useful benchmarking lesson in this workshop — flag it explicitly.** `np.empty(200 MB)` costs ~0 resident memory; touching one byte per 4 KB page (`big[::4096] = 1`) costs the full 200 MB. This is the mechanism behind a benchmarking gotcha students will hit repeatedly: *first-touch cost lands on the first iteration of a loop*, so "warm up, then time" isn't superstition — it's routing around page faults you don't want to measure. Tie this directly back to the GPU workshop's "run cells twice" advice students may have already seen; this cell is the mechanistic explanation for that advice.

**Virtual memory as "a lie the hardware maintains" is the framing to keep, not soften.** Every process believes it owns a private, contiguous address space starting near zero — two processes can both use address `0x5000` and touch completely different physical RAM. This is what makes C pointers safe to hand between processes without them stepping on each other, and it's worth an explicit callback to Intro to C if that workshop preceded this one.

**Scheduler jitter cell — set expectations before running it.** Asking for a 1 ms sleep and getting ~1.05 ms median with occasional larger outliers is *not* a bug in the demo; it's timer granularity plus scheduling delay, and the exact numbers will vary run-to-run and machine-to-machine (a loaded CI runner or shared VM will show much worse tails than a quiet laptop). The generalizable point: **the OS is why your benchmark numbers wobble**, not noise in your code — every "why is my timing not reproducible" question from earlier workshops has part of its answer here.

**Connective tissue for the whole session:** memory and scheduling are both instances of the same idea from the intro (multiplexing) — the OS is sharing one physical resource (RAM, cores) across many claimants by maintaining a convincing illusion of dedicated ownership, and that illusion always has a cost, paid either in page faults (memory) or context switches (scheduling).
</details>

## 3. Virtual Memory

💡 **Intuition.** Every process believes it owns the *entire* address space — a private, contiguous billboard of bytes starting at (almost) zero. It's a lie the hardware maintains: the **page table** maps each 4 KB *virtual* page to wherever the kernel actually parked it in RAM (or on disk!). The lie is what makes [C pointers](../../Intro_Programming/Intro_C.ipynb) safe to hand out: your address 0x5000 and my address 0x5000 are different physical bytes.

In [4]:
# Our own memory map: every region the kernel granted this process
with open("/proc/self/maps") as f:
    lines = f.readlines()
print(f"{len(lines)} mapped regions. A sample:")
for line in lines[:4]:  print(line.rstrip())
print("...")
for line in lines[-3:]: print(line.rstrip())

154 mapped regions. A sample:
003ff000-00400000 rw-p 00000000 103:02 12463125                          /home/jibby2k1/.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu/bin/python3.12
00400000-00421000 r--p 00001000 103:02 12463125                          /home/jibby2k1/.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu/bin/python3.12
00421000-00e02000 r-xp 00022000 103:02 12463125                          /home/jibby2k1/.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu/bin/python3.12
00e02000-013a1000 r--p 00a03000 103:02 12463125                          /home/jibby2k1/.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu/bin/python3.12
...
7a27e1f66000-7a27e1f68000 rw-p 00038000 103:02 97528910                  /usr/lib/x86_64-linux-gnu/ld-linux-x86-64.so.2
7ffd0a39f000-7ffd0a3c1000 rw-p 00000000 00:00 0                          [stack]
ffffffffff600000-ffffffffff601000 --xp 00000000 00:00 0                  [vsyscall]


**What just happened.** `/proc/self/maps` listed 154 separate memory regions for a process that hasn't done anything yet — the Python interpreter binary, `libpython`, shared libraries, the stack, and (last line) the `[vsyscall]` page. Each line is one entry in the page table's bookkeeping: a virtual address range, its permissions (`r`/`w`/`x`), and where it's backed (a file on disk, or nothing for anonymous memory like the stack). The takeaway isn't the exact count — it's that *before you've allocated a single Python object*, the kernel has already built a substantial map just to load the interpreter, and every one of those regions is virtual: real physical RAM is only committed when a page inside it is actually touched, which the next cell demonstrates directly.

In [5]:
# Allocation is lazy: address space is cheap, physical pages are charged on TOUCH
import numpy as np

def rss_mb():
    with open("/proc/self/status") as f:
        for line in f:
            if line.startswith("VmRSS"):
                return int(line.split()[1]) / 1024

before = rss_mb()
big = np.empty(200_000_000, dtype=np.uint8)      # "allocate" 200 MB
after_alloc = rss_mb()
big[::4096] = 1                                   # touch one byte per page
after_touch = rss_mb()
print(f"resident memory: before {before:.0f} MB → after np.empty {after_alloc:.0f} MB → after touching pages {after_touch:.0f} MB")
del big

resident memory: before 77 MB → after np.empty 77 MB → after touching pages 268 MB


`np.empty(200 MB)` cost almost nothing — the kernel handed out *promises*, not pages. Touching the pages forced it to deliver. Moral for benchmarking: **first-touch cost lands on your first iteration**, which is (one reason) why the GPU notebook told you to run cells twice.

## 4. The Scheduler

💡 **Intuition.** More runnable threads than cores ⇒ someone waits. The scheduler slices CPU time and swaps processes on and off cores; each swap (*context switch*) trashes caches and costs microseconds. Your timing jitter is mostly *other people's processes*.

In [6]:
import time

# Measure scheduler jitter: ask to sleep 1 ms, see what we actually get
gaps = []
for _ in range(200):
    t0 = time.perf_counter()
    time.sleep(0.001)
    gaps.append((time.perf_counter() - t0) * 1000)

import numpy as np
g = np.array(gaps)
print(f"asked for 1.000 ms → got median {np.median(g):.3f} ms, worst {g.max():.3f} ms")
print("that overshoot IS the operating system: timer granularity + scheduling delay")

asked for 1.000 ms → got median 1.053 ms, worst 1.128 ms
that overshoot IS the operating system: timer granularity + scheduling delay


---
### 🕐 Session 3 of 4 — *Concurrency in Practice* (~40 min)
**Goal:** spawn threads and processes; cause (and fix) a race condition; build a producer/consumer pipeline.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (the shell).

---

<details><summary>🎓 <b>Teacher notes — Session 3: Concurrency in Practice</b></summary>

**Sequence matters here: break it before you fix it.** The unsafe `worker` cell is deliberately rigged to make the race reliable (the `if v % 1000 == 0: time.sleep(0)` invites a context switch right in the read-modify-write gap) — without that nudge, this exact race might not show up for thousands of runs on a fast machine, which is *the point*: races are probabilistic, and "it worked when I tested it" is not evidence of correctness. Say this explicitly, because it's the single most important intuition about concurrency bugs: the bug doesn't get more present when you add a `print` to investigate it, the *timing* changes and the bug tends to hide (the classic "heisenbug").

**The lock cell isn't just "add a lock and it works" — narrate *why*.** `with lock:` forces the read-modify-write triplet (`v = counter`; sleep; `counter = v+1`) to execute as one atomic unit per thread — no other thread can interleave inside that block. Worth asking students to predict the output *before* running it, since the unsafe version primed them to expect loss.

**The GIL section is where Python-specific misconceptions live — correct them directly.** Threads in CPython buy *concurrency* (overlapping I/O waits) but not *parallelism* for CPU-bound pure-Python loops, because only one thread runs bytecode at a time. The multiprocessing cell right after is the fix for CPU-bound work — separate processes, separate memory, true parallelism, at the cost of no shared state (hence the producer/consumer *queue* pattern instead of shared variables in the next cell). If a student asks "why not just use threads for everything," this is the moment to answer.

**Producer/consumer with `queue.Queue` is the constructive alternative to locks**, not just another demo: no shared mutable state to protect means no race to have. Flag this as the pattern behind every real sensor-acquisition pipeline (a hardware thread reading samples, a processing thread consuming them) — this is the shape students will reach for in their capstone if it involves live data.
</details>

## 5. Threads, Races, Locks

💡 **Intuition.** A **race condition** is two workers doing read-modify-write on the same data with no coordination: both read 5, both write 6, one increment vanishes. The bug is *probabilistic* — it disappears when you look closely (add a print → timing changes → race hides). That's why the fix is discipline (locks), not debugging.

In [7]:
import threading

counter = 0
def worker(n_iters):
    global counter
    for _ in range(n_iters):
        v = counter                    # read
        # the gap between read and write is where disaster lives;
        # we widen it so the race shows up reliably in a demo
        if v % 1000 == 0: time.sleep(0)   # invite a context switch
        counter = v + 1                # write

threads = [threading.Thread(target=worker, args=(50_000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print(f"expected 200,000  —  got {counter:,}  (lost {200_000 - counter:,} updates!)")

expected 200,000  —  got 53,000  (lost 147,000 updates!)


In [8]:
counter = 0
lock = threading.Lock()
def safe_worker(n_iters):
    global counter
    for _ in range(n_iters):
        with lock:                     # one thread in here at a time
            v = counter
            if v % 1000 == 0: time.sleep(0)
            counter = v + 1

threads = [threading.Thread(target=safe_worker, args=(50_000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print(f"with the lock: {counter:,} — every update survived")

with the lock: 200,000 — every update survived


**What just happened.** The unsafe version above lost 147,000 of 200,000 increments (`53,000` survived) — four threads each doing `v = counter; ...; counter = v + 1` with no coordination means two threads can both read the same value of `counter` before either writes back, so one thread's increment silently overwrites the other's. Wrapping the read-modify-write in `with lock:` forces the whole triplet to run as one uninterruptible unit per thread, and the count comes back exact: `200,000`. Nothing about the *arithmetic* changed — only who's allowed to interleave when. This is also why the loss rate isn't deterministic or predictable from first principles: it depends on exactly how the scheduler happens to interleave the four threads, which is precisely the "probabilistic bug" property flagged above.

### 5.1. Threads vs Processes in Python

CPython's **GIL** lets only one thread run Python bytecode at a time — threads buy you *concurrency* (overlapping waits) but not *parallelism* for pure-Python math. For CPU-bound work, use **processes** (separate memory, true parallelism) — or NumPy, which releases the GIL inside its C loops. This is why the [GPU workshop's](../../Intro_GPU/Intro_GPU.ipynb) vectorization advice works even from single-threaded Python.

In [9]:
from multiprocessing import Pool

def heavy(seed):
    # pure-Python CPU work: sum of squares
    s = 0
    for i in range(5_000_000):
        s += (i * seed) % 7
    return s

for label, runner in [("serial", map), ("4 processes", None)]:
    tic = time.perf_counter()
    if runner:
        results = list(map(heavy, [1, 2, 3, 4]))
    else:
        with Pool(4) as pool:
            results = pool.map(heavy, [1, 2, 3, 4])
    print(f"{label:12s}: {time.perf_counter() - tic:.2f} s")

serial      : 0.48 s


4 processes : 0.15 s


### 5.2. Producer/Consumer

The queue is the honest way for concurrent workers to talk: no shared mutable state, no races — the pattern behind every data-acquisition pipeline (sensor thread produces, processing thread consumes).

In [10]:
import queue

q = queue.Queue(maxsize=8)                       # bounded: producer blocks if consumer lags
DONE = object()

def producer():
    rng = np.random.default_rng(0)
    for i in range(12):
        q.put(rng.standard_normal(4))            # "samples from the sensor"
    q.put(DONE)

def consumer(out):
    while (block := q.get()) is not DONE:
        out.append(np.sqrt(np.mean(block**2)))   # compute RMS per block

rms = []
t1, t2 = threading.Thread(target=producer), threading.Thread(target=consumer, args=(rms,))
t1.start(); t2.start(); t1.join(); t2.join()
print("processed", len(rms), "blocks; RMS of first 3:", np.round(rms[:3], 3))

processed 12 blocks; RMS of first 3: [0.337 0.868 0.788]


---
### 🕐 Session 4 of 4 — *The Shell & Automation* (~35 min)
**Goal:** compose tools with pipes; control processes; package an experiment so it reruns cleanly.
**Builds on:** Session 1.

---

<details><summary>🎓 <b>Teacher notes — Session 4: The Shell & Automation</b></summary>

**Open by naming the callback: the pipe is Session 3's producer/consumer queue, built into the kernel.** `ps -eo comm,rss | head -6 | awk ...` is three separate processes, each reading a stream and writing a stream, scheduled *concurrently* by the OS — the exact same bounded-handoff pattern as the `queue.Queue` demo, just at the process level instead of the thread level. This reframing is worth dwelling on: it turns "the shell is a weird DSL" into "the shell is the same concurrency primitive you already understand, with syntax."

**The `ps` output will vary wildly by machine — don't let students get hung up on matching the notebook's exact processes.** The teaching point is the *pipeline mechanics* (three independent programs stitched by the kernel), not that `java`/`firefox` will appear on their machine too.

**Redirection and exit codes are the automation glue, and the exit-code convention is worth stating explicitly:** `0` means success, anything else means failure, and `&&` chains commands on success. This is the entire contract that lets shell scripts, cron jobs, and CI pipelines make decisions — `grep -q ... && echo "target reached"` only prints if grep actually found the pattern.

**The reproducibility checklist (§6.2) is the session's real payoff — treat it as a checklist to apply, not just read.** If time allows, have students audit one of their own past scripts (from Intro to Python or the ML workshops) against the four points: pinned environment, config from argv/env, logs to files with nonzero exit on failure, results to a database. Most will fail at least one — that's the productive outcome, not a grading exercise.

**Close the loop to the whole workshop:** processes (S1), memory (S2), concurrency (S3), and the shell (S4) are the four layers every other notebook in this curriculum has been running on top of, invisibly, the entire time.
</details>

## 6. The Shell

💡 **Intuition.** The pipe `|` is the shell's superpower: each program reads a stream and writes a stream, and the kernel plumbs them together — tiny tools composing into pipelines, with the OS scheduling all stages *concurrently*. It's the producer/consumer queue from Session 3, built into the operating system.

In [11]:
%%bash
# Pipeline: of the 5 most memory-hungry processes, show name + resident MB
ps -eo comm,rss --sort=-rss | head -6 | awk 'NR>1 {printf "%-20s %6.0f MB\n", $1, $2/1024}' 

java                   2861 MB
firefox                1511 MB
Isolated                  0 MB
dropbox

                1021 MB
Isolated                  0 MB


In [12]:
%%bash
# Redirection + exit codes: the glue of automation
echo "run 1: loss=0.42" > results.log
echo "run 2: loss=0.35" >> results.log
grep -q "loss=0.35" results.log && echo "target reached (exit code $?)"
rm results.log

target reached (exit code 0)


### 6.1. Environment & Reproducibility

Environment variables are the OS-level configuration channel — the reason `conda activate` works and half of all "works on my machine" bugs exist.

In [13]:
%%bash
echo "PATH has $(echo $PATH | tr ':' '\n' | wc -l) entries; python resolves to: $(which python3)"
MY_EXPERIMENT_SEED=42 python3 -c "import os; print('the child saw seed', os.environ['MY_EXPERIMENT_SEED'])"

PATH has 13 entries; python resolves to: /usr/bin/python3


the child saw seed 42


**What just happened.** `PATH` resolved to 13 directories the shell searches in order to find `python3` — the first match wins, which is the entire mechanism behind "wrong Python got picked up" bugs when a `conda`/`venv` environment isn't activated (or is shadowed by something earlier in `PATH`). The second line shows environment variables crossing the process boundary from Session 1: `MY_EXPERIMENT_SEED=42` set in front of the command is inherited by the *child* process, which reads it back out of its own `os.environ`. Environment variables are a one-way, copy-on-fork channel — the parent shell's variable and the child's `os.environ['MY_EXPERIMENT_SEED']` are independent copies of the same string, not a shared reference; the child can't write back to the parent's environment. This is the OS-level mechanism underneath every `--seed`/`--lr` flag and every `conda activate` in the reproducibility checklist below.

### 6.2. The Reproducible-Experiment Checklist

Package every experiment so future-you can rerun it:

1. **Pin the environment** — `requirements.txt` / `environment.yml`, committed.
2. **Take config from argv/env, not edits** — `python train.py --lr 0.01 --seed 42` ([Intro to C §8](../../Intro_Programming/Intro_C.ipynb) taught the same interface).
3. **Log to files, exit nonzero on failure** — so shell scripts and cron can react.
4. **Write results to a database, not scattered CSVs** — the [Databases workshop](../Intro_Databases/Intro_Databases.ipynb) built exactly this logger.


## 7. Conclusion

Processes are running programs with kernel-side identity; memory is a per-process illusion maintained by page tables; schedulers cause your timing jitter; races die by lock or by queue; and the shell composes it all. You now know what's *underneath* every other workshop.

---
## Where next

- [Intro to Databases](../Intro_Databases/Intro_Databases.ipynb) — files + locks + transactions, industrialized.
- [Intro to GPU Systems](../../Intro_GPU/README.md) — a second processor with its own memory hierarchy to schedule.
- [Intro to C](../../Intro_Programming/Intro_C.ipynb) — the language the kernel itself is written in.